# 🧠 Trabajo Práctico Integrador: Clasificación con Decision Tree vs Random Forest
**Cátedra:** Inteligencia Artificial - 5º Nivel ISI (Ciclo Lectivo 2026)  
**Universidad Tecnológica Nacional (UTN)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielBenjaminLopez/IA-DecisionTree/blob/main/TPI_IA_Clasificacion.ipynb)

### 👥 Equipo de Desarrollo:
- Integrante 1 (Legajo) - Arquitectura y Algoritmo Decision Tree C4.5
- Integrante 2 (Legajo) - Preprocesamiento y Random Forest
- Integrante 3 (Legajo) - Interfaz de Usuario y Visualización
- Integrante 4 (Legajo) - Experimentación y Redacción de Informe LNCS


## 🎯 Objetivos del Trabajo Práctico
1. **Implementar desde cero el algoritmo Decision Tree (DT)** basado en las pautas teóricas del sistema **C4.5 de Quinlan** (capaz de manejar atributos continuos y categóricos, calculando Information Gain y Gain Ratio).
2. **Implementar y evaluar el modelo de ensamble Random Forest (RF)** utilizando .
3. **Comparar el rendimiento de ambos modelos** analizando métricas de clasificación (Accuracy, Precision, Recall, F1-Score, Matriz de Confusión) y tiempos de ejecución al variar hiperparámetros.
4. **Disponer de una interfaz amigable e interactiva** para ejecutar simulaciones fácilmente y visualizar los resultados obtenidos.


In [ ]:
# ==============================================================================
# 1. INSTALACIÓN Y CONFIGURACIÓN DEL ENTORNO
# ==============================================================================
import sys
import os
import time
import math
from collections import Counter

# Instalar librerías requeridas si no se encuentran en el entorno
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import confusion_matrix
    print("✅ Todas las dependencias están disponibles.")
except ImportError:
    print("Instalando dependencias necesarias...")
    !pip install -q pandas numpy matplotlib seaborn ipywidgets scikit-learn
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import confusion_matrix
    print("✅ Instalación completada con éxito.")


## 🌳 2. Implementación de Decision Tree (C4.5) desde Cero

El algoritmo implementado se fundamenta en las técnicas del sistema **C4.5 (Quinlan, 1993)**:
- **Atributos continuos:** Para cada atributo numérico, se ordenan sus valores y se evalúan como puntos de corte binarios ( \le 	heta$ frente a  > 	heta$) los puntos medios entre valores adyacentes donde cambia la clase.
- **Atributos categóricos:** Partición de ramas por cada categoría discreta observada.
- **Criterio de Razón de Ganancia (Gain Ratio):** Normaliza la ganancia respecto a la información de división para evitar el sesgo hacia atributos con muchas categorías:
24	ext{GainRatio}(S, A) = rac{	ext{Gain}(S, A)}{	ext{SplitInfo}(S, A)}24
donde:
24	ext{Gain}(S, A) = 	ext{Entropy}(S) - \sum_{v \in 	ext{Valores}(A)} rac{|S_v|}{|S|} 	ext{Entropy}(S_v)24
24	ext{SplitInfo}(S, A) = - \sum_{v \in 	ext{Valores}(A)} rac{|S_v|}{|S|} \log_2 rac{|S_v|}{|S|}24


In [ ]:
# ==============================================================================
# 2. CLASES NODE Y DECISION TREE C4.5
# ==============================================================================

class Node:
    """Representa un nodo en la jerarquía del Árbol de Decisión."""
    def __init__(self, feature=None, threshold=None, children=None, value=None,
                 is_continuous=False, probabilities=None, samples=0, entropy=0.0):
        self.feature = feature                  # Nombre de la característica evaluada
        self.threshold = threshold              # Umbral de corte (si es continuo)
        self.children = children or {}          # Ramas {valor: Node} o {'<=': Node, '>': Node}
        self.value = value                      # Clase predicha si es nodo hoja
        self.is_continuous = is_continuous      # Booleano indicador de división continua
        self.probabilities = probabilities or {}# Distribución de probabilidad de clases
        self.samples = samples                  # Muestras que alcanzaron el nodo
        self.entropy = entropy                  # Entropía del nodo

    @property
    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifierC45:
    """Clasificador C4.5 desarrollado íntegramente por el equipo."""
    def __init__(self, criterion="gain_ratio", max_depth=10, min_samples_split=2):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        self.feature_names = []
        self.classes_ = []

    def fit(self, X, y):
        X_df = pd.DataFrame(X).copy()
        self.feature_names = list(X_df.columns)
        y_list = list(y)
        self.classes_ = sorted(list(set(y_list)))
        self.root = self._build_tree(X_df.values.tolist(), y_list, depth=0)
        return self

    def predict(self, X):
        X_df = pd.DataFrame(X)[self.feature_names]
        return [self._predict_sample(self.root, row) for row in X_df.values.tolist()]

    def _build_tree(self, X, y, depth):
        n_samples = len(y)
        counts = Counter(y)
        majority_class = counts.most_common(1)[0][0]
        node_entropy = self._entropy(y)
        probabilities = {c: count / n_samples for c, count in counts.items()}

        # Criterios de parada
        if len(counts) <= 1 or depth >= self.max_depth or n_samples < self.min_samples_split:
            return Node(value=majority_class, probabilities=probabilities, samples=n_samples, entropy=node_entropy)

        best_split = self._find_best_split(X, y, node_entropy)
        if best_split is None or best_split["gain"] <= 1e-7:
            return Node(value=majority_class, probabilities=probabilities, samples=n_samples, entropy=node_entropy)

        feat_idx = best_split["feature_idx"]
        feat_name = self.feature_names[feat_idx]
        is_continuous = best_split["is_continuous"]
        children = {}

        if is_continuous:
            threshold = best_split["threshold"]
            left_X, left_y, right_X, right_y = [], [], [], []
            for row, target in zip(X, y):
                val = row[feat_idx]
                if val is not None and val <= threshold:
                    left_X.append(row); left_y.append(target)
                else:
                    right_X.append(row); right_y.append(target)

            if len(left_y) == 0 or len(right_y) == 0:
                return Node(value=majority_class, probabilities=probabilities, samples=n_samples, entropy=node_entropy)

            children["<="] = self._build_tree(left_X, left_y, depth + 1)
            children[">"] = self._build_tree(right_X, right_y, depth + 1)
            return Node(feature=feat_name, threshold=threshold, children=children, is_continuous=True,
                        probabilities=probabilities, samples=n_samples, entropy=node_entropy)
        else:
            partitions = {}
            for row, target in zip(X, y):
                cat = str(row[feat_idx])
                if cat not in partitions: partitions[cat] = ([], [])
                partitions[cat][0].append(row); partitions[cat][1].append(target)

            for cat, (sub_X, sub_y) in partitions.items():
                children[cat] = self._build_tree(sub_X, sub_y, depth + 1)
            return Node(feature=feat_name, children=children, is_continuous=False,
                        probabilities=probabilities, samples=n_samples, entropy=node_entropy)

    def _find_best_split(self, X, y, parent_entropy):
        best_score = -1.0
        best_split = None
        n_features = len(self.feature_names)
        n_samples = len(y)

        for feat_idx in range(n_features):
            col_vals = [row[feat_idx] for row in X]
            is_num = any(isinstance(v, (int, float)) and not isinstance(v, bool) for v in col_vals if v is not None)

            if is_num:
                pairs = sorted([(v, lbl) for v, lbl in zip(col_vals, y) if v is not None], key=lambda x: x[0])
                if len(pairs) < 2: continue

                for i in range(len(pairs) - 1):
                    v1, l1 = pairs[i]; v2, l2 = pairs[i + 1]
                    if v1 == v2 or l1 == l2: continue
                    threshold = (v1 + v2) / 2.0

                    left_y = [lbl for v, lbl in pairs if v <= threshold]
                    right_y = [lbl for v, lbl in pairs if v > threshold]
                    if not left_y or not right_y: continue

                    p_left = len(left_y) / n_samples
                    p_right = len(right_y) / n_samples
                    gain = parent_entropy - (p_left * self._entropy(left_y) + p_right * self._entropy(right_y))
                    split_info = -( (p_left * math.log2(p_left) if p_left > 0 else 0) +
                                    (p_right * math.log2(p_right) if p_right > 0 else 0) )
                    score = (gain / split_info) if (self.criterion == "gain_ratio" and split_info > 1e-7) else gain

                    if score > best_score:
                        best_score = score
                        best_split = {"feature_idx": feat_idx, "is_continuous": True, "threshold": threshold, "gain": gain, "score": score}
            else:
                categories = list(set([str(v) for v in col_vals]))
                if len(categories) <= 1: continue

                parts = {c: [] for c in categories}
                for v, lbl in zip(col_vals, y): parts[str(v)].append(lbl)

                rem_ent, split_info = 0.0, 0.0
                for c, sub_y in parts.items():
                    if sub_y:
                        p = len(sub_y) / n_samples
                        rem_ent += p * self._entropy(sub_y)
                        split_info -= p * math.log2(p)

                gain = parent_entropy - rem_ent
                score = (gain / split_info) if (self.criterion == "gain_ratio" and split_info > 1e-7) else gain

                if score > best_score:
                    best_score = score
                    best_split = {"feature_idx": feat_idx, "is_continuous": False, "threshold": None, "gain": gain, "score": score}

        return best_split

    def _entropy(self, labels):
        if not labels: return 0.0
        n = len(labels)
        return -sum((c / n) * math.log2(c / n) for c in Counter(labels).values() if c > 0)

    def _predict_sample(self, node, row):
        if node.is_leaf: return node.value
        feat_idx = self.feature_names.index(node.feature)
        val = row[feat_idx]
        if node.is_continuous:
            branch = "<=" if (val is not None and val <= node.threshold) else ">"
            if branch in node.children: return self._predict_sample(node.children[branch], row)
        else:
            cat = str(val)
            if cat in node.children: return self._predict_sample(node.children[cat], row)
        return max(node.probabilities.items(), key=lambda x: x[1])[0]

    def render_text(self, node=None, indent=""):
        if node is None: node = self.root
        lines = []
        if node.is_leaf:
            return [f"{indent}└── Predict: [{node.value}] (muestras={node.samples}, pureza={node.entropy:.3f})"]
        for branch, child in node.children.items():
            cond = f"{node.feature} {branch} {node.threshold:.2f}" if node.is_continuous else f"{node.feature} == '{branch}'"
            lines.append(f"{indent}├── IF {cond}")
            lines.extend(self.render_text(child, indent + "│   "))
        return lines


## 🌲 3. Random Forest (Scikit-Learn) y Módulo Evaluador

Para el modelo de ensamble **Random Forest**, utilizamos  conforme a lo estipulado en la consigna. Se implementa un preprocesador automático (codificación One-Hot de variables categóricas) y un evaluador de métricas unificado.


In [ ]:
# ==============================================================================
# 3. EVALUADOR Y ENTRENAMIENTO COMPARATIVO
# ==============================================================================

class ExperimentRunner:
    """Gestiona la ejecución de experimentos, cálculo de métricas y renderizado de resultados."""

    @staticmethod
    def evaluate_model(y_true, y_pred, elapsed_train, elapsed_infer):
        y_true, y_pred = list(y_true), list(y_pred)
        n = len(y_true)
        acc = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp) / n
        labels = sorted(list(set(y_true) | set(y_pred)))

        pos_label = labels[-1]
        tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == pos_label and yp == pos_label)
        fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt != pos_label and yp == pos_label)
        fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == pos_label and yp != pos_label)
        tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt != pos_label and yp != pos_label)

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0

        return {
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "Tiempo Entrenamiento (s)": elapsed_train,
            "Tiempo Inferencia (s)": elapsed_infer,
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "cm": confusion_matrix(y_true, y_pred, labels=labels),
            "labels": labels
        }


## 🎛️ 4. Interfaz Interactiva Amigable (Google Colab)

A continuación se presenta la interfaz interactiva desarrollada con . Permite:
1. **Seleccionar o cargar el dataset** ( u ).
2. **Ajustar hiperparámetros** para el Árbol C4.5 y Random Forest.
3. **Ejecutar la comparativa en un solo clic** y visualizar pestañas con tablas, matrices de confusión y la estructura del árbol.


In [ ]:
# ==============================================================================
# 4. CONSTRUCCIÓN DE LA INTERFAZ CON IPYWIDGETS
# ==============================================================================

# Detección de archivos CSV disponibles
data_dir = "data"
available_csvs = [f for f in os.listdir(data_dir) if f.endswith(".csv")] if os.path.exists(data_dir) else []
if not available_csvs:
    available_csvs = ["data/provisional_kaggle.csv"]

# Controles de Datos
w_dataset = widgets.Dropdown(
    options=available_csvs,
    value=available_csvs[0] if available_csvs else None,
    description="Dataset:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="400px")
)

w_test_size = widgets.FloatSlider(
    value=0.25, min=0.10, max=0.50, step=0.05,
    description="Test Split:",
    style={"description_width": "130px"},
    readout_format=".0%"
)

# Hiperparámetros Decision Tree
w_dt_criterion = widgets.Dropdown(
    options=[("Razón de Ganancia (C4.5)", "gain_ratio"), ("Ganancia de Información (ID3)", "information_gain")],
    value="gain_ratio",
    description="Criterio DT:",
    style={"description_width": "130px"}
)

w_dt_depth = widgets.IntSlider(
    value=5, min=1, max=15, step=1,
    description="Profundidad DT:",
    style={"description_width": "130px"}
)

w_dt_min_split = widgets.IntSlider(
    value=4, min=2, max=20, step=1,
    description="Min Split DT:",
    style={"description_width": "130px"}
)

# Hiperparámetros Random Forest
w_rf_estimators = widgets.IntSlider(
    value=100, min=10, max=300, step=10,
    description="Árboles en RF:",
    style={"description_width": "130px"}
)

w_rf_depth = widgets.IntSlider(
    value=7, min=1, max=20, step=1,
    description="Profundidad RF:",
    style={"description_width": "130px"}
)

btn_run = widgets.Button(
    description="🚀 Entrenar y Comparar Modelos",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="320px", height="42px")
)

out_results = widgets.Output()

def on_run_clicked(b):
    with out_results:
        clear_output(wait=True)
        csv_path = os.path.join(data_dir, w_dataset.value) if not w_dataset.value.startswith("data/") else w_dataset.value
        
        if not os.path.exists(csv_path):
            print(f"❌ No se encontró el archivo: {csv_path}")
            return

        df = pd.read_csv(csv_path)
        display(HTML(f"<h4>📊 Datos cargados: {df.shape[0]} registros, {df.shape[1]} columnas</h4>"))

        # Seleccionar target (última columna por defecto)
        target_col = df.columns[-1]
        X = df.drop(columns=[target_col])
        y = df[target_col]

        # Train/Test Split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=w_test_size.value, random_state=42, stratify=y
        )

        # 1. ENTRENAR DECISION TREE C4.5 (DESDE CERO)
        display(HTML("<b>Entrenando Decision Tree C4.5 propio...</b>"))
        dt = DecisionTreeClassifierC45(
            criterion=w_dt_criterion.value,
            max_depth=w_dt_depth.value,
            min_samples_split=w_dt_min_split.value
        )
        t0 = time.time()
        dt.fit(X_train, y_train)
        dt_train_time = time.time() - t0

        t0 = time.time()
        dt_preds = dt.predict(X_test)
        dt_infer_time = time.time() - t0

        dt_metrics = ExperimentRunner.evaluate_model(y_test, dt_preds, dt_train_time, dt_infer_time)

        # 2. ENTRENAR RANDOM FOREST (SCIKIT-LEARN)
        display(HTML("<b>Entrenando Random Forest (Scikit-Learn)...</b>"))
        # Preprocesamiento One-Hot para RF (sklearn requiere inputs estrictamente numéricos)
        X_train_encoded = pd.get_dummies(X_train, drop_first=True)
        X_test_encoded = pd.get_dummies(X_test, drop_first=True).reindex(columns=X_train_encoded.columns, fill_value=0)

        rf = RandomForestClassifier(
            n_estimators=w_rf_estimators.value,
            max_depth=w_rf_depth.value,
            random_state=42
        )
        t0 = time.time()
        rf.fit(X_train_encoded, y_train)
        rf_train_time = time.time() - t0

        t0 = time.time()
        rf_preds = rf.predict(X_test_encoded)
        rf_infer_time = time.time() - t0

        rf_metrics = ExperimentRunner.evaluate_model(y_test, rf_preds, rf_train_time, rf_infer_time)

        # 3. MOSTRAR RESULTADOS COMPARATIVOS
        display(HTML("<hr><h3>📈 Resultados Comparativos de Rendimiento</h3>"))
        
        res_df = pd.DataFrame({
            "Métrica": ["Exactitud (Accuracy)", "Precisión (Precision)", "Sensibilidad (Recall)", "F1-Score", "Tiempo Entrenamiento (s)", "Tiempo Inferencia (s)"],
            "Decision Tree (C4.5 Propio)": [
                f"{dt_metrics['Accuracy']*100:.2f}%",
                f"{dt_metrics['Precision']:.4f}",
                f"{dt_metrics['Recall']:.4f}",
                f"{dt_metrics['F1-Score']:.4f}",
                f"{dt_metrics['Tiempo Entrenamiento (s)']:.4f}s",
                f"{dt_metrics['Tiempo Inferencia (s)']:.4f}s"
            ],
            "Random Forest (Ensamble Sklearn)": [
                f"{rf_metrics['Accuracy']*100:.2f}%",
                f"{rf_metrics['Precision']:.4f}",
                f"{rf_metrics['Recall']:.4f}",
                f"{rf_metrics['F1-Score']:.4f}",
                f"{rf_metrics['Tiempo Entrenamiento (s)']:.4f}s",
                f"{rf_metrics['Tiempo Inferencia (s)']:.4f}s"
            ]
        })
        display(res_df.set_index("Métrica"))

        # Graficar Matrices de Confusión
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        sns.heatmap(dt_metrics["cm"], annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
                    xticklabels=dt_metrics["labels"], yticklabels=dt_metrics["labels"])
        axes[0].set_title(f"Matriz Confusión - DT (C4.5)
Acc: {dt_metrics['Accuracy']*100:.1f}%")
        axes[0].set_xlabel("Predicción"); axes[0].set_ylabel("Valor Real")

        sns.heatmap(rf_metrics["cm"], annot=True, fmt="d", cmap="Greens", cbar=False, ax=axes[1],
                    xticklabels=rf_metrics["labels"], yticklabels=rf_metrics["labels"])
        axes[1].set_title(f"Matriz Confusión - Random Forest
Acc: {rf_metrics['Accuracy']*100:.1f}%")
        axes[1].set_xlabel("Predicción"); axes[1].set_ylabel("Valor Real")
        plt.tight_layout()
        plt.show()

        # Estructura del Árbol C4.5
        with widgets.Output():
            pass
        tree_text = dt.render_text()
        print(f"
🌳 Estructura Jerárquica del Árbol C4.5 (Profundidad máxima configurada: {w_dt_depth.value}):")
        print("
".join(tree_text[:25]))
        if len(tree_text) > 25:
            print(f"... ({len(tree_text) - 25} ramas adicionales omitidas en la vista previa)")

btn_run.on_click(on_run_clicked)

# Agrupación visual en Tabs y Boxes
box_data = widgets.VBox([widgets.HTML("<b>1. Selección de Datos:</b>"), w_dataset, w_test_size])
box_dt = widgets.VBox([widgets.HTML("<b>2. Hiperparámetros Decision Tree (C4.5):</b>"), w_dt_criterion, w_dt_depth, w_dt_min_split])
box_rf = widgets.VBox([widgets.HTML("<b>3. Hiperparámetros Random Forest:</b>"), w_rf_estimators, w_rf_depth])

ui_panel = widgets.VBox([
    widgets.HTML("<h2>🎛️ Panel de Control de Experimentos</h2>"),
    widgets.HBox([box_data, box_dt, box_rf]),
    widgets.Box([btn_run], layout=widgets.Layout(margin="15px 0 10px 0")),
    out_results
])

display(ui_panel)
